# 02 - Train the ADL classifier

| Setting | Value |
| --- | --- |
| Internet | **Off** |
| Accelerator | **GPU** |

**Add Data before running:** the prepared dataset from notebook 01 (it carries `manifest.json`
and `code/`), plus the pretrained backbone as its own dataset. Nothing can be downloaded here.

Selection metric is **validation macro-F1**, never accuracy. ADL data is dominated by sitting;
an accuracy-selected model predicts the majority class and misses every clinically
interesting event.


In [ ]:
# Preflight. Every assertion fails in seconds; discovering a missing mount after a 40-minute
# epoch is exactly what this cell prevents.
import json, socket, sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Enable the GPU accelerator in the notebook settings.'
print('gpu:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, '| cuda:', torch.version.cuda)

def has_internet(host='raw.githubusercontent.com', port=443, timeout=3):
    try:
        socket.create_connection((host, port), timeout=timeout)
        return True
    except OSError:
        return False

print('internet:', 'ENABLED - turn it off to match the reproducible path' if has_internet() else 'disabled (correct)')


In [ ]:
# Resolve mounts. Edit these names to match your attached datasets.
DATASET_DIR = Path('/kaggle/input/adl-lowres-v1')
PRETRAINED_DIR = Path('/kaggle/input/videomae-base-offline')
RUN_DIR = Path('/kaggle/working/runs/adl-001')

missing = [str(p) for p in (DATASET_DIR, PRETRAINED_DIR) if not p.exists()]
assert not missing, f'attach these datasets via Add Data: {missing}'

MANIFEST = DATASET_DIR / 'manifest.json'
assert MANIFEST.exists(), f'no manifest.json in {DATASET_DIR}; re-run notebook 01'

sys.path.insert(0, str(DATASET_DIR / 'code' / 'src'))
sys.path.insert(0, str(DATASET_DIR / 'code' / 'train'))

commit_file = DATASET_DIR / 'code' / 'COMMIT'
print('code commit:', commit_file.read_text().strip() if commit_file.exists() else 'unknown')

manifest = json.loads(MANIFEST.read_text())
print(f"dataset: {manifest['name']} v{manifest['version']}, {manifest['n_records']} records")
print('split by:', manifest['split_by'], '| seed:', manifest['seed'])


In [ ]:
# Reuse the committed helpers so notebook and CLI runs behave identically.
from _offline_tracker import OfflineTracker
from train_activity import class_weights, seed_everything

SEED = 42
EPOCHS = 30
BATCH_SIZE = 8
ACCUM_STEPS = 4
LR = 1e-4
CLIP_FRAMES = 16

seed_everything(SEED)

records = manifest['records']
labels = sorted({r['label'] for r in records if 'label' in r})
label_to_index = {name: i for i, name in enumerate(labels)}
train_counts = {
    name: sum(1 for r in records if r.get('label') == name and r.get('split') == 'train')
    for name in labels
}
weights = class_weights(train_counts, labels)

print(f'{len(labels)} classes, effective batch {BATCH_SIZE * ACCUM_STEPS}')
for name, weight in zip(labels, weights):
    print(f'  {name:<26} n={train_counts[name]:<6} weight={weight:.3f}')


## Data

The dataset reads the manifest, not the filesystem. Globbing a directory silently changes the
training set whenever the mount changes, which makes two runs incomparable without anything
looking wrong.

Augmentation is asymmetric on purpose: train-time degradation (downscale, blur, low light)
simulates CCTV, validation stays clean so the metric stays interpretable.


In [ ]:
import numpy as np
from torch.utils.data import DataLoader, Dataset

class ClipDataset(Dataset):
    """Manifest-driven clip dataset.

    Expects each record's `path` to resolve under DATASET_DIR. Frames are decoded to a
    (T, H, W, C) uint8 array; swap in decord or torchvision.io as your prepared format requires.
    """

    def __init__(self, records, root, split, clip_frames, train):
        self.records = [r for r in records if r.get('split') == split]
        self.root = Path(root)
        self.clip_frames = clip_frames
        self.train = train

    def __len__(self):
        return len(self.records)

    def _load_clip(self, record):
        path = self.root / record['path']
        if path.suffix == '.npy' and path.exists():
            frames = np.load(path)
        elif path.exists():
            import cv2
            capture = cv2.VideoCapture(str(path))
            collected = []
            while True:
                ok, frame = capture.read()
                if not ok:
                    break
                collected.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            capture.release()
            frames = np.asarray(collected) if collected else np.zeros((1, 224, 224, 3), np.uint8)
        else:
            # Synthetic fallback so the loop is verifiable before real clips are staged.
            frames = np.random.randint(0, 255, (self.clip_frames, 224, 224, 3), dtype=np.uint8)

        # Uniform temporal sampling: random offset while training, centred while validating.
        total = len(frames)
        if total >= self.clip_frames:
            span = total - self.clip_frames
            start = np.random.randint(0, span + 1) if (self.train and span > 0) else span // 2
            indices = np.arange(start, start + self.clip_frames)
        else:
            indices = np.clip(np.arange(self.clip_frames), 0, total - 1)
        return frames[indices]

    def __getitem__(self, index):
        record = self.records[index]
        clip = self._load_clip(record).astype(np.float32) / 255.0
        if self.train:
            if np.random.rand() < 0.5:
                clip = clip[:, :, ::-1]
            # Gamma jitter approximates the weak evening lighting that dominates real footage.
            clip = np.clip(clip ** np.random.uniform(0.7, 1.4), 0.0, 1.0)
        tensor = torch.from_numpy(np.ascontiguousarray(clip)).permute(3, 0, 1, 2)
        return tensor, label_to_index[record['label']]

train_set = ClipDataset(records, DATASET_DIR, 'train', CLIP_FRAMES, train=True)
val_set = ClipDataset(records, DATASET_DIR, 'val', CLIP_FRAMES, train=False)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'train clips: {len(train_set)}, val clips: {len(val_set)}')


In [ ]:
# Model. Loads strictly from the mounted weights directory; there is no online fallback, so a
# missing mount raises instead of quietly training from random initialisation.
import torch.nn as nn

def build_model(num_classes, pretrained_dir):
    try:
        from transformers import VideoMAEForVideoClassification
        model = VideoMAEForVideoClassification.from_pretrained(
            str(pretrained_dir),
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
            local_files_only=True,
        )
        return model, 'videomae'
    except Exception as error:
        print('VideoMAE unavailable:', error)
        print('falling back to a small 3D CNN so the loop stays runnable')
        model = nn.Sequential(
            nn.Conv3d(3, 32, 3, stride=(1, 2, 2), padding=1), nn.BatchNorm3d(32), nn.ReLU(),
            nn.Conv3d(32, 64, 3, stride=2, padding=1), nn.BatchNorm3d(64), nn.ReLU(),
            nn.Conv3d(64, 128, 3, stride=2, padding=1), nn.BatchNorm3d(128), nn.ReLU(),
            nn.AdaptiveAvgPool3d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, num_classes),
        )
        return model, 'cnn3d_fallback'

model, backbone_name = build_model(len(labels), PRETRAINED_DIR)
model = model.cuda()
print('backbone:', backbone_name)
print('parameters:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')


In [ ]:
# Train. AMP plus gradient accumulation, checkpointing on validation macro-F1.
tracker = OfflineTracker(RUN_DIR, {
    'task': 'adl_classification',
    'dataset': manifest['name'],
    'dataset_version': manifest['version'],
    'backbone': backbone_name,
    'labels': labels,
    'train_counts': train_counts,
    'hyperparameters': {
        'seed': SEED, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'accum_steps': ACCUM_STEPS, 'lr': LR, 'clip_frames': CLIP_FRAMES,
    },
})

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32).cuda(), label_smoothing=0.05
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=max(1, EPOCHS * (len(train_loader) // ACCUM_STEPS))
)
scaler = torch.cuda.amp.GradScaler()

def logits_of(output):
    return output.logits if hasattr(output, 'logits') else output

def macro_f1(confusion):
    """Unweighted mean of per-class F1, so a rare class counts as much as a common one."""
    scores = []
    for k in range(len(confusion)):
        tp = confusion[k, k]
        fp = confusion[:, k].sum() - tp
        fn = confusion[k, :].sum() - tp
        denominator = 2 * tp + fp + fn
        scores.append(0.0 if denominator == 0 else float(2 * tp / denominator))
    return float(np.mean(scores)), scores

best_f1 = -1.0
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running = 0.0
    for step, (clips, targets) in enumerate(train_loader):
        clips, targets = clips.cuda(non_blocking=True), targets.cuda(non_blocking=True)
        with torch.cuda.amp.autocast():
            loss = criterion(logits_of(model(clips)), targets) / ACCUM_STEPS
        scaler.scale(loss).backward()
        running += float(loss) * ACCUM_STEPS
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler.last_epoch < scheduler.total_steps - 1:
                scheduler.step()

    model.eval()
    confusion = np.zeros((len(labels), len(labels)), dtype=np.int64)
    with torch.no_grad(), torch.cuda.amp.autocast():
        for clips, targets in val_loader:
            predictions = logits_of(model(clips.cuda())).argmax(1).cpu().numpy()
            for truth, prediction in zip(targets.numpy(), predictions):
                confusion[truth, prediction] += 1

    val_f1, per_class = macro_f1(confusion)
    accuracy = float(np.trace(confusion) / max(1, confusion.sum()))
    tracker.log(epoch, train_loss=running / max(1, len(train_loader)), val_macro_f1=val_f1, val_accuracy=accuracy)
    print(f'epoch {epoch:02d}  loss {running / max(1, len(train_loader)):.4f}  macro-F1 {val_f1:.4f}  acc {accuracy:.4f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(
            {'model': model.state_dict(), 'labels': labels, 'epoch': epoch, 'val_macro_f1': val_f1},
            RUN_DIR / 'best.pt',
        )
        print(f'  checkpoint saved (macro-F1 {val_f1:.4f})')

print('best macro-F1:', best_f1)


In [ ]:
# Per-class report plus the confusion pairs that matter clinically.
final_f1, per_class = macro_f1(confusion)
print(f"{'class':<28}{'F1':>8}{'support':>10}")
for name, score in sorted(zip(labels, per_class), key=lambda item: item[1]):
    support = int(confusion[label_to_index[name], :].sum())
    print(f'{name:<28}{score:>8.3f}{support:>10}')

print()
print('Worst confusions (true -> predicted):')
pairs = [
    (int(confusion[i, j]), labels[i], labels[j])
    for i in range(len(labels)) for j in range(len(labels)) if i != j and confusion[i, j] > 0
]
for count, truth, predicted in sorted(pairs, reverse=True)[:8]:
    print(f'  {truth} -> {predicted}: {count}')

# resting vs lying and sitting vs transfer are the pairs that corrupt inactivity baselines,
# so they are called out rather than left to be spotted in a heatmap.
for a, b in [('resting', 'lying'), ('sitting', 'transfer'), ('meal', 'drink')]:
    if a in label_to_index and b in label_to_index:
        i, j = label_to_index[a], label_to_index[b]
        print(f'  critical pair {a}/{b}: {int(confusion[i, j])} + {int(confusion[j, i])} errors')

tracker.summarise(
    best_val_macro_f1=best_f1,
    final_val_macro_f1=final_f1,
    per_class_f1=dict(zip(labels, per_class)),
    confusion=confusion.tolist(),
    exit_criterion={'adl_macro_f1': '>= 0.70', 'met': bool(best_f1 >= 0.70)},
    backbone=backbone_name,
)
print()
print('run directory:', RUN_DIR)
print('exit criterion macro-F1 >= 0.70:', 'MET' if best_f1 >= 0.70 else 'NOT MET')


## If the exit criterion is not met

In rough order of expected value:

1. **Cut the label set.** Twelve ADL classes at 0.55 macro-F1 is worse for caregivers than six
   classes at 0.80. Merge the pairs listed as critical confusions above.
2. **Add elderly-subject data.** Models trained on young actors transfer badly to slower,
   assisted movement. Toyota Smarthome matters more here than more Kinetics.
3. **Lengthen the clip.** Meal and medication routines are not visible in 16 frames.
4. **Only then change the backbone.** Architecture is rarely the binding constraint at this
   data scale.

Do not raise the confidence threshold to make the numbers look better: that trades recall on
the rare classes, which are the ones the system exists to catch.
